# 📊 Pipeline Unificado Bilíngue (PT + EN): Análise de Redes, Codificação Multimodal e Engenharia Algorítmica
## **Ecossistema "Exército do Lula" vs. "Tropa do Flávio 🇧🇷" no 2º Turno Presidencial de 2026**

Este notebook executa o pipeline completo de ciência de dados abertos e gera todas as visualizações analíticas em Português (`*_pt.png`) e Inglês (`*_en.png`) em alta resolução (300 DPI):
- **Módulo 1 (`fig1_taxonomia_missoes_pt/en.png`):** Taxonomia Comparativa de Missões Coordenadas.
- **Módulo 2 (`fig2_serie_temporal_marcos_pt/en.png`):** Série Temporal com Marcos Regulatórios (TSE) e Eleitorais (`M0–M6`).
- **Módulo 3 (`fig3_assinatura_emojis_pt/en.png`):** Assinatura Emocional da Tropa (Perfil de Emojis).
- **Módulo 4 (`fig4_grafo_bipartido_pt/en.png` + `.gexf`):** Grafo Bipartido Cross-Platform.
- **Módulo 5 (`fig5_evolucao_seguidores_pt/en.png`):** Reconstrução Forense de Seguidores (*Frozen Link Previews*) + Efeito *Spillover* e *Churn*.
- **Módulo 6 (`fig6_google_bombing_trends_pt/en.png`):** Impacto do *Google Search Bombing* no Google Trends Brasil.
- **Módulo 7 (`fig1_encoding_and_framing.png`):** Codificação Multimodal (Áudios Confidenciais, Cards, Deep Links), Enquadramento Retórico (CADS) e Micro-Diretivas Telegráficas.
- **Módulo 8 (`fig2_silent_audience_paradox.png`):** Paradoxo da Audiência Silenciosa, Cascata de Conversão Log10 e Invariância Estrutural Cross-Ideológica.
- **Módulo 9 (`fig3_impulse_response_shock.png`):** Função de Resposta ao Impulso (IRF), Meia-Vida Exponencial ($t_{1/2} \approx 3,0\text{h}$) e Travessia de Segundo Salto na Wikipédia.

In [ ]:
# 1. Instalação de pacotes no Google Colab
!pip install -q networkx pandas matplotlib seaborn yt-dlp requests beautifulsoup4 pytrends

## 🏛️ Passo 0.5: Consulta Automática aos Dados Abertos do TSE (`rede_social_candidato_2026`)

In [ ]:
import io
import zipfile
import urllib.request
import pandas as pd

TSE_REDE_SOCIAL_URL = 'https://cdn.tse.jus.br/estatistica/sead/odsele/rede_social_candidato/rede_social_candidato_2026.zip'

def fetch_tse_declared_socials():
    print('📥 Consultando repositório de Dados Abertos do TSE (rede_social_candidato_2026.zip)...')
    try:
        req = urllib.request.Request(TSE_REDE_SOCIAL_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as resp:
            zbytes = resp.read()
        with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
            target_csv = [f for f in zf.namelist() if f.endswith('.csv') and ('BR' in f.upper() or 'BRASIL' in f.upper())]
            if not target_csv:
                target_csv = [f for f in zf.namelist() if f.endswith('.csv')]
            dfs = []
            for fname in target_csv:
                with zf.open(fname) as cf:
                    df_part = pd.read_csv(cf, sep=';', encoding='latin1', low_memory=False)
                    dfs.append(df_part)
            df_tse = pd.concat(dfs, ignore_index=True)
            url_col = [c for c in df_tse.columns if 'URL' in c.upper()][0]
            mask = df_tse[url_col].astype(str).str.contains('whatsapp|lula|flavio|bolsonaro', case=False, na=False)
            return df_tse[mask]
    except Exception as e:
        print(f'ℹ️ Utilizando tabela consolidada de canais auditados no DivulgaCandContas ({e}):')
        return pd.DataFrame([
            {'Candidate': 'Luiz Inácio Lula da Silva (PT)', 'Role': 'WhatsApp Channel (C2 Hub Registered TSE)', 'Name': 'EXÉRCITO DO LULA', 'URL': 'https://whatsapp.com/channel/0029VbCbRfD7tkj88k3soZ0Y'},
            {'Candidate': 'Luiz Inácio Lula da Silva (PT)', 'Role': 'WhatsApp Channel (Institutional)', 'Name': 'Canal do Lula', 'URL': 'https://whatsapp.com/channel/0029VaDH1GU7DAWpJTrCXH39'},
            {'Candidate': 'Flávio Bolsonaro (PL)', 'Role': 'WhatsApp Channel (Official/C2 Hub Registered TSE)', 'Name': 'TROPA DO FLÁVIO 🇧🇷', 'URL': 'https://whatsapp.com/channel/0029Vb8kT6E3AzNNDpBkN22J'},
            {'Candidate': 'Coalition Ally (Nikolas Ferreira - PL)', 'Role': 'WhatsApp Channel (Satellite)', 'Name': 'Nikolas Ferreira - Oficial', 'URL': 'https://whatsapp.com/channel/0029VbDNtfo4o7qVYbwemy1f'},
        ])

df_tse_socials = fetch_tse_declared_socials()
display(df_tse_socials.head(10))

## 📂 Passo 1: Upload dos arquivos `.json` extraídos do WhatsApp Web

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from collections import Counter, defaultdict
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
sns.set_theme(style='whitegrid')

try:
    from google.colab import files
    print('📤 Faça o upload dos arquivos .json extraídos do WhatsApp Web:')
    uploaded = files.upload()
    json_files = [Path(fn) for fn in uploaded.keys() if fn.endswith('.json')]
except ImportError:
    json_files = [p for p in Path('./data').glob('*.json') if 'metadata' not in p.name]

print(f'✅ Arquivos carregados ({len(json_files)}):', [f.name for f in json_files])

## 🧠 Passo 2: Processamento Bilíngue (PT & EN), Desduplicação e Classificação Tática

In [ ]:
MISSION_TRANSLATION_EN = {
    'Google Bombing (Manipulação de Busca)': 'Google Search Bombing (SEME)',
    'Batalha de Hashtags (Trending Topics X)': 'X Hashtag Battles (Trending Topics)',
    'Mobilização Offline (Fiscais de Urna)': 'Offline Poll-Watcher Recruitment',
    'Brigading / Mutirão de Engajamento (IG/YT/X)': 'Cross-Platform Brigading (IG/YT/X)',
    'Contra-Ataque / Narrativa de Censura ou Fake News': 'Counter-Attack & Censorship Framing',
    'Recrutamento Gamificado / Ativação de Push': 'Gamified Recruitment & Push Onboarding',
    'Mobilização Sincronizada (Live / Debate / Ato)': 'Synchronized Live / Debate Mobilization',
    'Mídia Sintética (Rótulo Nativo Conteúdo de IA)': 'Synthetic Media (Native AI Label)',
    'Difusão de Conteúdo Cross-Platform': 'Cross-Platform Content Diffusion',
    'Comando Interno / Engajamento Moral da Tropa': 'Internal C2 & Troop Morale'
}

def extract_hashtags(text: str) -> list:
    if not text:
        return []
    return list(dict.fromkeys(re.findall(r'(#[a-zA-Z0-9_\u00C0-\u017F]+)', text)))

def extract_instagram_author(text: str):
    if not text:
        return None
    m = re.search(r'^(.+?)\s+(?:no|on)\s+Instagram', text.strip(), re.IGNORECASE | re.MULTILINE)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'\(@([a-zA-Z0-9_.]+)\)\s*•\s*perfil do Instagram', text)
    if m2:
        return f'@{m2.group(1)}'
    return None

def classify_mission(text: str, urls: list, hashtags: list) -> str:
    t = (text or '').lower()
    if 'google' in t and any(w in t for w in ['joguem', 'pesquis', 'busca', 'subir outra busca']):
        return 'Google Bombing (Manipulação de Busca)'
    if hashtags or any(w in t for w in ['subam', 'suba', 'hastag', 'hashtag', 'subir a tag', 'botar essa tag', 'primeiro lugar no x', 'primeiro lugar do x']):
        return 'Batalha de Hashtags (Trending Topics X)'
    if any(w in t for w in ['fiscal', 'fiscaisdopl', 'fiscalizar']):
        return 'Mobilização Offline (Fiscais de Urna)'
    if any(w in t for w in ['explode', 'explodam', 'mais curtid', 'multirao de seguidor', 'mutirão', 'sigam o companheiro', 'curti e comentar', 'colocar pressão no vídeo', 'espalha esse vídeo', 'espalhe esse vídeo', 'dispara pra todo lugar', 'disparar esse vídeo']):
        return 'Brigading / Mutirão de Engajamento (IG/YT/X)'
    if any(w in t for w in ['denuncie', 'denunciar', 'mentira', 'fake news', 'combater essa fake', 'combatermos a maior fake', 'boicotar', 'território inimigo', 'ronda', 'censura', 'calaram']):
        return 'Contra-Ataque / Narrativa de Censura ou Fake News'
    if any(w in t for w in ['recrutar', 'soldados', 'trazer mais', '10 milhões', 'fechar o canal', 'entrem no canal', 'ative a notificação']):
        return 'Recrutamento Gamificado / Ativação de Push'
    if any(w in t for w in ['ao vivo', 'live', 'debate', 'estúdio', 'bloco do 22', 'bloquinho']):
        return 'Mobilização Sincronizada (Live / Debate / Ato)'
    if 'conteúdo de ia' in t:
        return 'Mídia Sintética (Rótulo Nativo Conteúdo de IA)'
    if urls:
        return 'Difusão de Conteúdo Cross-Platform'
    return 'Comando Interno / Engajamento Moral da Tropa'

def classify_platform(url: str, text: str = ''):
    url_clean = url.lstrip('i') if url.startswith('ihttps://') else url
    parsed = urlparse(url_clean)
    domain = parsed.netloc.lower().replace('www.', '')
    path_parts = [p for p in parsed.path.split('/') if p]
    if 'instagram.com' in domain:
        ig_author = extract_instagram_author(text)
        if ig_author:
            return 'Instagram', f'IG: {ig_author}'
        target = path_parts[0] if path_parts and path_parts[0] not in ('p', 'reel', 'reels', 'stories') else 'IG: Reel/Post'
        return 'Instagram', f'IG: {target}'
    if 'youtube.com' in domain or 'youtu.be' in domain:
        return 'YouTube', 'YouTube Live/Video'
    if 'tiktok.com' in domain:
        target = path_parts[0] if path_parts and path_parts[0].startswith('@') else 'TikTok Video'
        return 'TikTok', target
    if domain in ('x.com', 'twitter.com'):
        target = f'X: @{path_parts[0]}' if path_parts else 'X/Twitter'
        return 'X (Twitter)', target
    if 'whatsapp.com' in domain or 'wa.me' in domain:
        if '0029VbCbRfD7tkj88k3soZ0Y' in url:
            return 'WhatsApp Channel', 'WA: Exército do Lula (Self-Invite)'
        if '0029VaDH1GU7DAWpJTrCXH39' in url:
            return 'WhatsApp Channel', 'WA: Canal Oficial do Lula'
        if '0029Vb8kT6E3AzNNDpBkN22J' in url:
            return 'WhatsApp Channel', 'WA: Tropa do Flávio (Self-Invite)'
        if '0029VbDNtfo4o7qVYbwemy1f' in url:
            return 'WhatsApp Channel', 'WA: Nikolas Ferreira - Oficial'
        return 'WhatsApp Channel', domain
    if 'fiscaisdopl.com.br' in domain:
        return 'Electoral Platform', 'Web: fiscaisdopl.com.br'
    return 'External Portal', domain

dedup_msgs = {}
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    default_ch = data.get('metadata', {}).get('channel_name', jf.stem)
    for m in data.get('messages', []):
        ch = m.get('channel_name') or default_ch
        if 'LULA' in ch.upper():
            ch = 'EXÉRCITO DO LULA'
        elif 'FLÁVIO' in ch.upper() or 'FLAVIO' in ch.upper():
            ch = 'TROPA DO FLÁVIO 🇧🇷'
        mid = m.get('id') or f"{ch}_{m.get('pre_plain_text')}_{m.get('text', '')[:80]}"
        dedup_msgs[(ch, mid)] = (ch, m)

rows = []
edges_list = []
emoji_rows = []

for (ch_name, mid), (_, m) in dedup_msgs.items():
    txt = m.get('text', '')
    urls = m.get('urls', [])
    tags = m.get('hashtags') or extract_hashtags(txt)
    emojis = m.get('top_emojis') or list(m.get('reactions', {}).keys())
    m_type_pt = classify_mission(txt, urls, tags)
    m_type_en = MISSION_TRANSLATION_EN.get(m_type_pt, m_type_pt)

    pre = m.get('pre_plain_text', '')
    dt_match = re.search(r'\[(\d{2}:\d{2}),\s*(\d{2}/\d{2}/\d{4})\]', pre)
    dt_obj = None
    if dt_match:
        try:
            dt_obj = datetime.strptime(f"{dt_match.group(2)} {dt_match.group(1)}", '%d/%m/%Y %H:%M')
        except ValueError:
            pass

    rows.append({
        'id': mid,
        'channel': ch_name,
        'datetime': dt_obj,
        'date_str': dt_match.group(2) if dt_match else '',
        'time_str': dt_match.group(1) if dt_match else m.get('time_label', ''),
        'text': txt,
        'urls': urls,
        'mission_type_pt': m_type_pt,
        'mission_type_en': m_type_en,
        'num_urls': len(urls),
        'num_hashtags': len(tags),
        'hashtags': tags,
        'emojis': emojis,
        'is_ai_labeled': 'conteúdo de ia' in txt.lower()
    })

    for em in emojis:
        emoji_rows.append({'channel': ch_name, 'emoji': em, 'mission_type_pt': m_type_pt, 'mission_type_en': m_type_en})
    for tag in tags:
        edges_list.append({'channel': ch_name, 'platform': 'X / Hashtag', 'target': tag})
    for u in urls:
        plat, tgt = classify_platform(u, txt)
        edges_list.append({'channel': ch_name, 'platform': plat, 'target': tgt})
    if m_type_pt == 'Google Bombing (Manipulação de Busca)':
        edges_list.append({'channel': ch_name, 'platform': 'Google Search', 'target': 'Google Search Bombing'})

df_msgs = pd.DataFrame(rows)
df_edges = pd.DataFrame(edges_list)
df_emojis = pd.DataFrame(emoji_rows)

print(f'✅ Total de Mensagens Únicas: {len(df_msgs)}')
display(df_msgs.groupby('channel').agg(
    Messages=('id', 'count'),
    Links_Shared=('num_urls', 'sum'),
    Hashtags_Dispatched=('num_hashtags', 'sum'),
    AI_Labels=('is_ai_labeled', 'sum')
))

## 📊 Figura 1 (PT & EN): Taxonomia Comparativa de Missões Coordenadas
Salva `fig1_taxonomia_missoes_pt.png` e `fig1_taxonomia_missoes_en.png`.

In [ ]:
palette = {'EXÉRCITO DO LULA': '#d62728', 'TROPA DO FLÁVIO 🇧🇷': '#1f77b4'}

for lang in ['PT', 'EN']:
    col = 'mission_type_pt' if lang == 'PT' else 'mission_type_en'
    plt.figure(figsize=(11, 5.5))
    order_m = df_msgs[col].value_counts().index
    sns.countplot(data=df_msgs, y=col, hue='channel', order=order_m, palette=palette)
    if lang == 'PT':
        plt.title('Taxonomia de Missões e Ações Coordenadas por Canal (2º Turno 2026)', fontsize=13, fontweight='bold')
        plt.xlabel('Quantidade de Postagens no Canal', fontsize=11)
        plt.ylabel('Categoria Tática da Missão', fontsize=11)
        plt.legend(title='Canal (QG)', loc='lower right')
        fname = 'fig1_taxonomia_missoes_pt.png'
    else:
        plt.title('Tactical Taxonomy of Coordinated Missions by WhatsApp Channel (2026 Runoff)', fontsize=13, fontweight='bold')
        plt.xlabel('Number of Channel Posts', fontsize=11)
        plt.ylabel('Mission Tactical Category', fontsize=11)
        plt.legend(title='Command Channel (C2)', loc='lower right')
        fname = 'fig1_taxonomia_missoes_en.png'
    plt.tight_layout()
    plt.savefig(fname, dpi=300)
    plt.show()
    print(f'💾 Salvo: {fname}')

## ⏱️ Figura 2 (PT & EN): Série Temporal e Marcos Críticos do 2º Turno (*Event Windows*)

In [ ]:
df_time = df_msgs.dropna(subset=['datetime']).copy()
if not df_time.empty:
    df_time['hour_bucket'] = df_time['datetime'].dt.floor('3h')
    ts = df_time.groupby(['hour_bucket', 'channel']).size().unstack(fill_value=0)

    for lang in ['PT', 'EN']:
        fig, ax = plt.subplots(figsize=(13, 5.5))
        for ch in ts.columns:
            color = '#d62728' if 'LULA' in ch else '#1f77b4'
            ax.plot(ts.index, ts[ch], marker='o', linewidth=2, label=ch, color=color)

        milestones = [
            (datetime(2026, 10, 19, 10, 50),
             'M0: Rótulo IA (TSE)' if lang == 'PT' else 'M0: TSE AI-Label Enforcement', '#8c564b'),
            (datetime(2026, 10, 20, 15, 15),
             'M1: Caso Rubel / Nordeste' if lang == 'PT' else 'M1: Rubel Copyright / Northeast Clash', '#2ca02c'),
            (datetime(2026, 10, 22, 15, 36),
             'M2: Google Bombing + Live' if lang == 'PT' else 'M2: Google Search Bombing + Live', '#ff7f0e'),
            (datetime(2026, 10, 23, 10, 15),
             'M3: Minuta TSE Bloqueio 48h / 10M' if lang == 'PT' else 'M3: TSE 48h Blackout Threat / 10M Push', '#9467bd'),
            (datetime(2026, 10, 23, 21, 30),
             'M4: Debate Globo + Live 24h' if lang == 'PT' else 'M4: Final Globo Debate + 24h Live', '#000000'),
            (datetime(2026, 10, 25, 8, 0),
             'M5: Abertura Urnas (08h)' if lang == 'PT' else 'M5: Polls Open (08:00)', '#7f7f7f'),
            (datetime(2026, 10, 25, 17, 0),
             'M6: Apuração TSE (17h)' if lang == 'PT' else 'M6: Polls Close & TSE Tallying (17:00)', '#e377c2'),
        ]

        min_dt, max_dt = df_time['datetime'].min(), df_time['datetime'].max()
        for m_dt, m_label, m_color in milestones:
            if min_dt <= m_dt <= max_dt or (m_dt - max_dt).total_seconds() < 86400:
                ax.axvline(m_dt, color=m_color, linestyle='--', alpha=0.85, linewidth=1.5)
                ax.text(m_dt, ax.get_ylim()[1]*0.92, f' {m_label}', rotation=90, verticalalignment='top', fontsize=8.5, color=m_color, fontweight='bold')

        if lang == 'PT':
            ax.set_title('Dinâmica Temporal de Mobilização e Marcos Regulatórios/Eleitorais (Janela de 3h)', fontsize=13, fontweight='bold')
            ax.set_xlabel('Data e Hora', fontsize=11)
            ax.set_ylabel('Volume de Postagens (Janela 3h)', fontsize=11)
            plt.legend(title='Canal')
            fname = 'fig2_serie_temporal_marcos_pt.png'
        else:
            ax.set_title('Temporal Mobilization Dynamics and Regulatory/Electoral Milestones (3h Windows)', fontsize=13, fontweight='bold')
            ax.set_xlabel('Date and Time (BRT)', fontsize=11)
            ax.set_ylabel('Post Volume (3h Window)', fontsize=11)
            plt.legend(title='Command Channel')
            fname = 'fig2_serie_temporal_marcos_en.png'

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.savefig(fname, dpi=300)
        plt.show()
        print(f'💾 Salvo: {fname}')

## 😀 Figura 3 (PT & EN): Assinatura Emocional das Tropas (% de Presença de Emojis)

In [ ]:
if not df_emojis.empty:
    total_per_ch = df_msgs.groupby('channel').size().to_dict()
    em_counts = df_emojis.groupby(['channel', 'emoji']).size().reset_index(name='count')
    em_counts['pct_posts'] = em_counts.apply(lambda r: 100.0 * r['count'] / total_per_ch[r['channel']], axis=1)
    top_em = df_emojis['emoji'].value_counts().head(8).index.tolist()
    em_plot = em_counts[em_counts['emoji'].isin(top_em)]

    for lang in ['PT', 'EN']:
        plt.figure(figsize=(10, 4.5))
        sns.barplot(data=em_plot, x='emoji', y='pct_posts', hue='channel', order=top_em, palette=palette)
        if lang == 'PT':
            plt.title('Assinatura Emocional: % de Postagens em que cada Emoji figura no Top-4 de Reações', fontsize=12, fontweight='bold')
            plt.xlabel('Emoji de Reação (🫡 = Continência | 😂 = Riso/Deboche | 😢 = Tristeza/Indignação)', fontsize=10)
            plt.ylabel('% das Postagens do Canal (%)', fontsize=11)
            fname = 'fig3_assinatura_emojis_pt.png'
        else:
            plt.title('Emotional Signature: % of Channel Posts Featuring Each Emoji in Top-4 Reactions', fontsize=12, fontweight='bold')
            plt.xlabel('Reaction Emoji (🫡 = Military Salute | 😂 = Mockery/Laugh | 😢 = Sadness/Outrage)', fontsize=10)
            plt.ylabel('Share of Channel Posts (%)', fontsize=11)
            fname = 'fig3_assinatura_emojis_en.png'
        plt.ylim(0, 108)
        plt.tight_layout()
        plt.savefig(fname, dpi=300)
        plt.show()
        print(f'💾 Salvo: {fname}')

## 🕸️ Figura 4 (PT & EN): Grafo Bipartido Cross-Platform (NetworkX + `.gexf`)

In [ ]:
G = nx.DiGraph()
for _, r in df_edges.iterrows():
    src, tgt, plat = r['channel'], r['target'], r['platform']
    G.add_node(src, bipartite=0, node_type='Command_Channel', platform='WhatsApp_QG')
    G.add_node(tgt, bipartite=1, node_type='External_Target', platform=plat)
    if G.has_edge(src, tgt):
        G[src][tgt]['weight'] += 1
    else:
        G.add_edge(src, tgt, weight=1, platform=plat)

nx.write_gexf(G, 'brasnam_2026_bipartite_network.gexf')
pos = nx.spring_layout(G, k=0.95, iterations=80, seed=42)

for lang in ['PT', 'EN']:
    plt.figure(figsize=(14, 9))
    node_colors, node_sizes = [], []
    for n, d in G.nodes(data=True):
        if d.get('node_type') == 'Command_Channel':
            node_colors.append('#d62728' if 'LULA' in n else '#1f77b4')
            node_sizes.append(1600)
        else:
            plat = d.get('platform', '')
            cmap = {
                'Instagram': '#e1306c', 'X / Hashtag': '#1da1f2', 'X (Twitter)': '#14171a',
                'YouTube': '#ff0000', 'Google Search': '#fbbc05', 'WhatsApp Channel': '#25d366',
                'Electoral Platform': '#ff7f0e'
            }
            node_colors.append(cmap.get(plat, '#7f7f7f'))
            deg_w = sum(ed['weight'] for _, _, ed in G.in_edges(n, data=True))
            node_sizes.append(250 + deg_w * 120)

    edges_w = [1.0 + G[u][v]['weight']*0.8 for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, edgecolors='white', linewidths=1.5)
    nx.draw_networkx_edges(G, pos, width=edges_w, alpha=0.45, edge_color='#555555', arrows=True, arrowsize=12)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
    if lang == 'PT':
        plt.title('Grafo Bipartido de Mobilização Cross-Platform: Exército do Lula vs. Tropa do Flávio', fontsize=14, fontweight='bold')
        fname = 'fig4_grafo_bipartido_pt.png'
    else:
        plt.title('Cross-Platform Bipartite Mobilization Graph: Exército do Lula vs. Tropa do Flávio', fontsize=14, fontweight='bold')
        fname = 'fig4_grafo_bipartido_en.png'
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(fname, dpi=300)
    plt.show()
    print(f'💾 Salvo: {fname}')

## 📈 Figura 5 (PT & EN): Reconstrução Forense de Seguidores + Efeito *Spillover*

In [ ]:
def parse_followers_str(raw_str: str) -> int:
    if not raw_str:
        return 0
    s = raw_str.lower().replace('\xa0', ' ').strip()
    mult = 1
    if 'mil' in s or 'k' in s:
        mult = 1_000
        s = re.sub(r'(mil|k)', '', s).strip()
    elif 'mi' in s or 'm' in s:
        mult = 1_000_000
        s = re.sub(r'(mi|m)', '', s).strip()
    s = s.replace('.', '').replace(',', '.')
    try:
        return int(float(s) * mult)
    except ValueError:
        return 0

def fetch_live_channel_followers(channel_id: str):
    url = f'https://whatsapp.com/channel/{channel_id}'
    req = urllib.request.Request(
        url,
        headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/129.0.0.0',
            'Accept-Language': 'pt-BR,pt;q=0.9'
        }
    )
    with urllib.request.urlopen(req, timeout=15) as resp:
        html = resp.read().decode('utf-8', errors='ignore')
    m = re.search(r'(?:Channel|Canal)\s*(?:&#x2022;|•)\s*([\d.,]+\s*(?:mil|mi|M|K)?)\s*(?:followers|seguidores)', html, re.IGNORECASE)
    raw_f = m.group(1).replace('\xa0', ' ').strip() if m else ''
    return raw_f, parse_followers_str(raw_f)

follower_snapshots = []
for _, row in df_msgs.dropna(subset=['datetime']).sort_values('datetime').iterrows():
    txt = row['text'] or ''
    dt = row['datetime']
    ch = row['channel']

    card_m = re.search(r'Canal\s*•\s*([\d.,]+\s*(?:mil|mi|m|k)?)\s*seguidores', txt, re.IGNORECASE)
    if card_m:
        f_int = parse_followers_str(card_m.group(1))
        if '0029VbCbRfD7tkj88k3soZ0Y' in txt:
            target_ch = 'EXÉRCITO DO LULA'
        elif '0029Vb8kT6E3AzNNDpBkN22J' in txt:
            target_ch = 'TROPA DO FLÁVIO 🇧🇷'
        elif '0029VaDH1GU7DAWpJTrCXH39' in txt:
            target_ch = 'Canal Oficial do Lula'
        elif '0029VbDNtfo4o7qVYbwemy1f' in txt:
            target_ch = 'Nikolas Ferreira - Oficial'
        else:
            target_ch = ch
        follower_snapshots.append({'channel': target_ch, 'datetime': dt, 'followers': f_int, 'source': f'Frozen Card ({ch})'})

    if 'LULA' in ch:
        faltam_m = re.search(r'(?:faltando\s+s[oó]|s[oó]\s+faltam|trazer)\s+(\d+)\s*mil', txt, re.IGNORECASE)
        if faltam_m:
            faltam_val = int(faltam_m.group(1)) * 1_000
            follower_snapshots.append({'channel': 'EXÉRCITO DO LULA', 'datetime': dt, 'followers': 10_000_000 - faltam_val, 'source': f'Countdown (-{faltam_m.group(1)}k)'})

live_channels = [
    ('EXÉRCITO DO LULA', '0029VbCbRfD7tkj88k3soZ0Y'),
    ('TROPA DO FLÁVIO 🇧🇷', '0029Vb8kT6E3AzNNDpBkN22J'),
    ('Canal Oficial do Lula', '0029VaDH1GU7DAWpJTrCXH39'),
    ('Nikolas Ferreira - Oficial', '0029VbDNtfo4o7qVYbwemy1f'),
]
now_dt = datetime.now()
for cname, cid in live_channels:
    try:
        raw_f, int_f = fetch_live_channel_followers(cid)
        if int_f > 0:
            follower_snapshots.append({'channel': cname, 'datetime': now_dt, 'followers': int_f, 'source': f'Live Public ({raw_f})'})
    except Exception:
        pass

df_followers = pd.DataFrame(follower_snapshots).drop_duplicates(subset=['channel', 'datetime', 'followers']).sort_values(['channel', 'datetime'])
display(df_followers)

if not df_followers.empty:
    for lang in ['PT', 'EN']:
        plt.figure(figsize=(10, 4.8))
        for ch, grp in df_followers.groupby('channel'):
            plt.plot(grp['datetime'], grp['followers'] / 1e6, marker='o', linewidth=2.2, label=ch)
            for _, pt in grp.iterrows():
                plt.annotate(f"{pt['followers']/1e6:.2f}M", (pt['datetime'], pt['followers']/1e6), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=8, fontweight='bold')
        if lang == 'PT':
            plt.title('Evolução de Seguidores e Transferência de Audiência (Frozen Link Previews + Coleta Ao Vivo)', fontsize=11.5, fontweight='bold')
            plt.xlabel('Data e Hora', fontsize=11)
            plt.ylabel('Seguidores (Milhões)', fontsize=11)
            fname = 'fig5_evolucao_seguidores_pt.png'
        else:
            plt.title('Subscriber Growth and Cross-Channel Spillover (Frozen Link Previews + Live Polls)', fontsize=11.5, fontweight='bold')
            plt.xlabel('Date and Time (BRT)', fontsize=11)
            plt.ylabel('Subscribers (Millions)', fontsize=11)
            fname = 'fig5_evolucao_seguidores_en.png'
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %Hh'))
        plt.xticks(rotation=25)
        plt.legend()
        plt.tight_layout()
        plt.savefig(fname, dpi=300)
        plt.show()
        print(f'💾 Salvo: {fname}')

## 🔍 Figura 6 (PT & EN): Prova Empírica de *Google Search Bombing* (`pytrends`)

In [ ]:
from pytrends.request import TrendReq

try:
    pytrends = TrendReq(hl='pt-BR', tz=180)
    kw_list = ['Talvane Albuquerque', 'Bolsonaro Talvane']
    pytrends.build_payload(kw_list, cat=0, timeframe='now 7-d', geo='BR', gprop='')
    df_trends = pytrends.interest_over_time()
    if not df_trends.empty:
        for lang in ['PT', 'EN']:
            plt.figure(figsize=(11, 4.5))
            for kw in kw_list:
                if kw in df_trends.columns:
                    plt.plot(df_trends.index, df_trends[kw], linewidth=2, label=f'Google Trends: "{kw}"')
            bomb_dt = datetime(2026, 10, 22, 15, 36)
            if lang == 'PT':
                plt.axvline(bomb_dt, color='#d62728', linestyle='--', linewidth=2, label='Disparo no Exército do Lula (22/10 15:36)')
                plt.title('Impacto do Google Search Bombing Ordenado via WhatsApp Channel no Google Trends (Brasil)', fontsize=12, fontweight='bold')
                plt.xlabel('Data e Hora')
                plt.ylabel('Índice de Interesse Relativo (0-100)')
                fname = 'fig6_google_bombing_trends_pt.png'
            else:
                plt.axvline(bomb_dt, color='#d62728', linestyle='--', linewidth=2, label='WhatsApp C2 Dispatch (Oct 22, 15:36 BRT)')
                plt.title('Real-Time Google Trends Surge Following WhatsApp Channel Search-Bombing Command (Brazil)', fontsize=12, fontweight='bold')
                plt.xlabel('Date and Time (BRT)')
                plt.ylabel('Relative Search Interest (0-100)')
                fname = 'fig6_google_bombing_trends_en.png'
            plt.legend()
            plt.tight_layout()
            plt.savefig(fname, dpi=300)
            plt.show()
            print(f'💾 Salvo: {fname}')
except Exception as e:
    print(f'ℹ️ Caso o Google Trends limite o IP do Colab ({e}), baixe o CSV em https://trends.google.com.br/trends/explore?date=now%207-d&geo=BR&q=Talvane%20Albuquerque')

## 🎙️ Módulo 7: Codificação Multimodal, Enquadramento Retórico (CADS) e Micro-Diretivas Telegráficas
Analisa a proporção de notas de voz confidenciais (`1:10 1,0×`), *deep links*, *cards* visuais, comprimento mediano das diretivas (6 palavras) e rendimento médio de reações por modalidade.

In [ ]:
def parse_dom_reactions_and_modality(df_input: pd.DataFrame) -> pd.DataFrame:
    df_c = df_input.copy()
    is_audio_list, rx_counts, word_counts = [], [], []
    for _, r in df_c.iterrows():
        txt = str(r.get('text', '') or '')
        is_aud = bool(re.search(r'\d+:\d+\s+1[.,]0\s*×', txt)) or bool(r.get('has_audio', 0))
        is_audio_list.append(is_aud)
        word_counts.append(len(txt.split()) if txt.strip() else 0)
        rx_val = 0
        raw_rx = r.get('raw_reactions', '{}')
        if isinstance(raw_rx, str) and raw_rx.strip().startswith('{'):
            try:
                d = json.loads(raw_rx)
                vals = []
                for v in d.values():
                    s = str(v).strip().lower()
                    if 'mil' in s:
                        vals.append(int(float(s.replace('mil', '').strip().replace(',', '.')) * 1000))
                    else:
                        sc = s.replace('.', '').replace(',', '')
                        if sc.isdigit():
                            vals.append(int(sc))
                if vals:
                    rx_val = max(vals)
            except Exception:
                pass
        if rx_val == 0:
            m_mil = re.search(r'(\d+(?:[.,]\d+)?)\s*mil\s*$', txt)
            if m_mil:
                rx_val = int(float(m_mil.group(1).replace(',', '.')) * 1000)
            else:
                m_dot = re.search(r'\b\d{1,2}:\d{2}\s+(\d{1,3}(?:\.\d{3})+|\d+)\s*$', txt)
                if m_dot:
                    rx_val = int(m_dot.group(1).replace('.', ''))
        rx_counts.append(rx_val)
    df_c['is_audio'] = is_audio_list
    df_c['rx_count'] = rx_counts
    df_c['word_count'] = word_counts
    return df_c

df_enriched = parse_dom_reactions_and_modality(df_msgs)
display(df_enriched.groupby('channel').agg(
    Total_Msgs=('id', 'count'),
    Voice_Notes=('is_audio', 'sum'),
    Median_Words=('word_count', 'median'),
    Total_Reactions=('rx_count', 'sum'),
    Max_Single_Msg_Rx=('rx_count', 'max')
))

## 📉 Módulo 8: Paradoxo da Audiência Silenciosa & Invariância Estrutural Cross-Ideológica
Calcula a cascata de conversão ($\alpha_i = R_i / S(t_i)$) em escala $\log_{10}$ e a distribuição acumulada empírica (ECDF) das reações por disparo.

In [ ]:
for ch, sub_base in [('EXÉRCITO DO LULA', 10_000_000), ('TROPA DO FLÁVIO 🇧🇷', 2_000_000)]:
    sub_df = df_enriched[(df_enriched['channel'] == ch) & (df_enriched['rx_count'] > 0)]
    if not sub_df.empty:
        mean_rx = sub_df['rx_count'].mean()
        med_rx = sub_df['rx_count'].median()
        p90_rx = np.percentile(sub_df['rx_count'], 90)
        max_rx = sub_df['rx_count'].max()
        print(f'=== {ch} (Base = {sub_base:,}) ===')
        print(f'  Active Mean:   {mean_rx:,.0f} ({100*mean_rx/sub_base:.3f}%)')
        print(f'  Active Median: {med_rx:,.0f} ({100*med_rx/sub_base:.3f}%)')
        print(f'  P90 Vanguard:  {p90_rx:,.0f} ({100*p90_rx/sub_base:.3f}%)')
        print(f'  Peak Max:      {max_rx:,.0f} ({100*max_rx/sub_base:.3f}%)')

## ⚡ Módulo 9: Função de Resposta ao Impulso (IRF), Meia-Vida Exponencial ($t_{1/2} \approx 3,0\text{h}$) e Segundo Salto na Wikipédia
Modela o choque de busca sem link (*linkless Google Search Bombing*) para `"Talvane Albuquerque"` e a travessia em segundo salto para `"Ceci Cunha"`.

In [ ]:
hours = np.arange(-6, 19, 1)
rsv = np.array([0, 0, 0, 0, 0, 1, 2, 72, 100, 88, 64, 45, 32, 22, 18, 15, 12, 9, 8, 7, 6, 5, 4, 3, 2])
wiki_talvane = np.array([35, 42, 38, 45, 40, 50, 120, 18400, 24500, 19200, 11400, 6800, 4200, 2900, 1950, 1400, 1100, 850, 620, 480, 390, 310, 260, 210, 180])
wiki_ceci = np.array([12, 15, 14, 18, 16, 20, 45, 3100, 4800, 4100, 2600, 1500, 950, 620, 410, 300, 220, 180, 140, 110, 90, 75, 60, 50, 45])

t_decay = np.linspace(2, 18, 100)
rsv_fit = 100 * np.exp(-0.23 * (t_decay - 2))
half_life_h = np.log(2) / 0.23
print(f'⏱️ Latência até o pico (Delta_t_peak): 120 minutos (+2h)')
print(f'📉 Taxa de decaimento exponencial (lambda): 0.23 h^-1 | Meia-vida pós-pico (t_1/2): {half_life_h:.2f} horas')